# Cache aside
Read from cache first, then load and store missing data.


In [ ]:
# Cache-aside reads cache first, then fills it from the source of truth.
cache: dict[str, str] = {}
database = {"task:1": "Ship API"}

def get_task(key: str) -> str:
    if key not in cache:
        cache[key] = database[key]
    return cache[key]

print(get_task("task:1"))


## Polished version
Keep cache and provider behind interfaces and generate stable keys from request data.


In [ ]:
# Interfaces let production use Redis while tests use an in-memory cache.
import hashlib
from dataclasses import dataclass
from typing import Protocol

@dataclass(frozen=True)
class ChatRequest:
    prompt: str

@dataclass(frozen=True)
class ChatResponse:
    text: str

class Provider(Protocol):
    async def complete(self, request: ChatRequest) -> ChatResponse: ...

class Cache(Protocol):
    async def get(self, key: str) -> ChatResponse | None: ...
    async def set(self, key: str, value: ChatResponse) -> None: ...

class FakeProvider:
    async def complete(self, request: ChatRequest) -> ChatResponse:
        return ChatResponse(request.prompt.upper())

class MemoryCache:
    def __init__(self) -> None:
        self.values: dict[str, ChatResponse] = {}
    async def get(self, key: str) -> ChatResponse | None:
        return self.values.get(key)
    async def set(self, key: str, value: ChatResponse) -> None:
        self.values[key] = value

class ChatService:
    def __init__(self, provider: Provider, cache_store: Cache) -> None:
        self.provider = provider
        self.cache = cache_store

    async def complete(self, request: ChatRequest) -> ChatResponse:
        # A stable digest creates the same safe key for the same request.
        key = hashlib.sha256(request.prompt.encode()).hexdigest()
        cached = await self.cache.get(key)
        if cached is not None:
            return cached
        # Only call the expensive provider after a cache miss.
        response = await self.provider.complete(request)
        await self.cache.set(key, response)
        return response

service = ChatService(FakeProvider(), MemoryCache())
print(await service.complete(ChatRequest("hello")))
